# Chapter 4 — Analysis

Runs the numerical experiment

**Outputs** (into `OUT_DIR`)

| file | contents |
|---|---|
| `results_over_k_all_orbits.csv` | one row per map, observable, orbit and window length `k` |
| `ci_curves_<map>_<obs>_orb1.parquet` | per-candidate fits and confidence intervals, one orbit |


In [ ]:
!pip install -q numpy pandas scipy statsmodels pyarrow tqdm
print("dependencies ready")

In [ ]:
import glob
import math
import os
import time
import warnings
from dataclasses import dataclass
from typing import Callable, Dict, List, Optional, Tuple

import numpy as np
import pandas as pd
from scipy import stats
from scipy.stats import genextreme, norm
from statsmodels.tools.sm_exceptions import InterpolationWarning
from statsmodels.tsa.stattools import adfuller, kpss
from tqdm.auto import tqdm

print("imports ok")

## Configuration

Everything adjustable lives here.

In [ ]:
# --- experiment configuration ---------------------------------------
N = 30_000                 # series length used for selection
N_ORBITS = 30
K_MAX = 10
I_STEP = 5                 # candidate grid spacing
SEED = 546123
ALPHA = 0.05
I_MIN = 5
N_MIN = 30                 # minimum block maxima; bounds i_max
I_REF = 5
NOISE_STD = 1e-6           # observational noise; required for the doubling map
TRANSIENT = 100
EPS_HESS = 1e-4
EPS_JAC = 1e-6
CI_ORBIT_ID = 0            # orbit used for the appendix CI curves
OUT_DIR = "ch4_results"
K_VALUES = tuple(range(1, K_MAX + 1))


# progress bars

print(f"n={N}, orbits={N_ORBITS}, k=1..{K_MAX}, grid step={I_STEP}, out={OUT_DIR}")

## Systems and observables

The doubling and logistic maps, the local expansion rate at the maximising
point, the three observable classes, orbit generation with the observational
noise channel, the moving-minimum transformation and block maxima.

The `1e-6` noise is required for the doubling map: it is implemented in finite
precision and loses a bit per iterate, so without it the numerical orbit
degenerates after a few dozen steps. It sits far below the resolution of the
block maxima.

In [ ]:
# ============================================================
# Reproducibility
# ============================================================
def set_seed(seed: int = 546123) -> np.random.Generator:
    np.random.seed(seed)
    return np.random.default_rng(seed)


# ============================================================
# Maps + derivatives
# ============================================================
def doubling_map(x: float) -> float:
    return (2.0 * x) % 1.0


def doubling_lambda_at(_: float) -> float:
    return 2.0


def logistic_map(x: float) -> float:
    return 4.0 * x * (1.0 - x)


def logistic_lambda_at(p: float) -> float:
    # |T'(x)| = |4 - 8x|
    return float(abs(4.0 - 8.0 * p))


# ============================================================
# Distance and observables
# ============================================================
def dist_to_p(x: np.ndarray, p: float) -> np.ndarray:
    return np.abs(x - p)


def phi_frechet(
    x: np.ndarray,
    p: float,
    alpha: float = 0.3,
    eps: float = 1e-12,
) -> np.ndarray:
    if alpha <= 0:
        raise ValueError("For Frechet observable, alpha must be > 0.")
    d = np.maximum(dist_to_p(x, p), eps)
    return d ** (-alpha)


def phi_gumbel(x: np.ndarray, p: float, eps: float = 1e-12) -> np.ndarray:
    d = np.maximum(dist_to_p(x, p), eps)
    return -np.log(d)


def phi_weibull(
    x: np.ndarray,
    p: float,
    xi: float = -0.3,
    C: float = 1.0,
    eps: float = 1e-12,
) -> np.ndarray:
    if xi >= 0:
        raise ValueError("For Weibull observable, xi must be < 0.")
    alpha = -xi
    d = np.maximum(dist_to_p(x, p), eps)
    return C - d ** alpha


# ============================================================
# Moving minimum (window k)
# ============================================================
def moving_minimum(y: np.ndarray, k: int) -> np.ndarray:
    if k <= 1:
        return y.copy()
    n = y.shape[0]
    m = n - k + 1
    if m <= 0:
        raise ValueError("k too large for series length.")
    out = np.empty(m, dtype=float)
    for i in range(m):
        out[i] = np.min(y[i:i + k])
    return out


# ============================================================
# Block maxima
# ============================================================
def block_maxima(y: np.ndarray, block_size: int) -> np.ndarray:
    n = y.shape[0]
    m = n // block_size
    if m <= 0:
        return np.array([], dtype=float)
    y_trunc = y[: m * block_size]
    return y_trunc.reshape(m, block_size).max(axis=1)


# ============================================================
# Orbit generation
# ============================================================
def generate_orbit(
    T: Callable[[float], float],
    n: int,
    transient: int,
    rng: np.random.Generator,
    noise_std: float,
) -> np.ndarray:
    x = float(rng.random())

    for _ in range(transient):
        x = T(x) + rng.normal(0.0, noise_std)
        x = float(np.clip(x, 0.0, 1.0))

    out = np.empty(n, dtype=float)
    for t in range(n):
        x = T(x) + rng.normal(0.0, noise_std)
        x = float(np.clip(x, 0.0, 1.0))
        out[t] = x

    return out

## Step 4 — the reparameterisation

Makes the GEV estimates invariant in the block multiple. Composes the block
maxima scaling with the moving-minimum scaling
`g(k, T) = lam ** (-(k - 1) * xi)`.

In [ ]:
# ============================================================
# Revised reparameterisation
# ============================================================
def rescale_params(
    mu_j: float,
    sigma_j: float,
    xi: float,
    j: float,
    lam: float,
    k: int,
) -> Tuple[float, float, float]:
    """Reparameterise GEV estimates to be invariant in the block multiple j.

    Implements Step 4 of Algorithm 1 (Theorem: revised reparameterised block
    scaling) exactly, for all three tail classes:

      xi > 0 :  sigma* = lam^{-(k-1)xi} sigma_j j^{-xi}
                mu*    = lam^{-(k-1)xi} [mu_j - sigma_j (j^xi - 1)/(xi j^xi)]

      xi < 0 :  sigma* = lam^{-(k-1)xi} sigma_j j^{-xi}
                mu*    = [mu_j - sigma_j (j^xi - 1)/(xi j^xi)]
                         + (sigma_j j^{-xi} / xi) (lam^{-(k-1)xi} - 1)

      xi = 0 :  sigma* = sigma_j
                mu*    = mu_j - sigma_j log j + sigma_j log(lam^{-(k-1)})
    """
    if j <= 0:
        raise ValueError("j must be positive.")
    if lam <= 0:
        raise ValueError("lam must be positive.")
    if k < 1:
        raise ValueError("k must be at least 1.")
    if sigma_j <= 0 or (not np.isfinite([mu_j, sigma_j, xi]).all()):
        return (float("nan"), float("nan"), float("nan"))

    # -----------------------------
    # Gumbel case
    # -----------------------------
    if abs(xi) < 1e-10:
        mu_base = mu_j - sigma_j * math.log(j)
        log_factor = -(k - 1) * math.log(lam)
        mu_star = mu_base + sigma_j * log_factor
        return float(mu_star), float(sigma_j), 0.0

    # -----------------------------
    # Remove block aggregation
    # -----------------------------
    sigma_base = sigma_j * (j ** (-xi))
    mu_base = mu_j - sigma_base * ((j ** xi - 1.0) / xi)

    # -----------------------------
    # Moving-minimum.
    # -----------------------------
    factor = lam ** (-(k - 1) * xi)

    if xi > 0:
        # Frechet: 
        mu_star = factor * mu_base
        sigma_star = factor * sigma_base
    else:
        # Weibull: 
        mu_star = mu_base + (sigma_base / xi) * (factor - 1.0)
        sigma_star = factor * sigma_base

    return float(mu_star), float(sigma_star), float(xi)

## GEV fitting, goodness of fit and delta-method intervals

In [ ]:
# ============================================================
# GEV fit + KS
# ============================================================
@dataclass
class GEVFit:
    c: float
    loc: float
    scale: float
    xi: float  # xi = -c


def fit_gev_mle(z: np.ndarray) -> Optional[GEVFit]:
    if len(z) < 10:
        return None
    c, loc, scale = genextreme.fit(z)
    if not np.isfinite([c, loc, scale]).all() or scale <= 0:
        return None
    return GEVFit(c=float(c), loc=float(loc), scale=float(scale), xi=float(-c))


def ks_pvalue_gev(z: np.ndarray, fit: GEVFit) -> float:
    _, p = stats.kstest(z, "genextreme", args=(fit.c, fit.loc, fit.scale))
    return float(p)


# ============================================================
# Stationarity tests
# ============================================================
def adf_pvalue(z: np.ndarray) -> float:
    try:
        return float(adfuller(z, autolag="AIC")[1])
    except Exception:
        return 1.0


def kpss_pvalue(z: np.ndarray) -> float:
    try:
        with warnings.catch_warnings():
            warnings.simplefilter("ignore", category=InterpolationWarning)
            stat, p, lags, crit = kpss(z, regression="c", nlags="auto")
        if (not np.isfinite(p)) or p <= 0.0:
            return 1e-6
        return float(p)
    except Exception:
        return 0.0


# ============================================================
# Delta-method CI for raw + rescaled parameters
# ============================================================
def _gev_nll_clog(z: np.ndarray, x: np.ndarray) -> float:
    # x = [c, loc, log_scale]
    c = float(x[0])
    loc = float(x[1])
    log_scale = float(x[2])
    if not np.isfinite([c, loc, log_scale]).all():
        return float("inf")
    scale = math.exp(log_scale)
    if scale <= 0 or (not np.isfinite(scale)):
        return float("inf")
    ll = genextreme.logpdf(z, c=c, loc=loc, scale=scale)
    if not np.isfinite(ll).all():
        return float("inf")
    return float(-np.sum(ll))


def _numerical_hessian(f, x: np.ndarray, eps: float) -> np.ndarray:
    x = np.asarray(x, dtype=float)
    n = x.size
    H = np.zeros((n, n), dtype=float)
    fx = f(x)

    for i in range(n):
        ei = np.zeros(n)
        ei[i] = 1.0

        f_ip = f(x + eps * ei)
        f_im = f(x - eps * ei)
        H[i, i] = (f_ip - 2.0 * fx + f_im) / (eps ** 2)

        for j2 in range(i + 1, n):
            ej = np.zeros(n)
            ej[j2] = 1.0
            f_pp = f(x + eps * ei + eps * ej)
            f_pm = f(x + eps * ei - eps * ej)
            f_mp = f(x - eps * ei + eps * ej)
            f_mm = f(x - eps * ei - eps * ej)
            hij = (f_pp - f_pm - f_mp + f_mm) / (4.0 * eps ** 2)
            H[i, j2] = hij
            H[j2, i] = hij

    return H


def _jacobian_g_x(
    z: np.ndarray,
    x0: np.ndarray,
    j: float,
    lam: float,
    k: int,
    eps: float,
) -> np.ndarray:
    """
    Jacobian of g(x) where x=(c,loc,log_scale) and g maps to
    (mu*, sigma*, xi*).
    """
    def g(x: np.ndarray) -> np.ndarray:
        c = float(x[0])
        loc = float(x[1])
        scale = math.exp(float(x[2]))
        xi = -c
        mu_star, sigma_star, xi_star = rescale_params(loc, scale, xi, j=j, lam=lam, k=k)
        return np.array([mu_star, sigma_star, xi_star], dtype=float)

    g0 = g(x0)
    J = np.zeros((3, 3), dtype=float)
    for col in range(3):
        dx = np.zeros(3, dtype=float)
        dx[col] = eps
        g1 = g(x0 + dx)
        J[:, col] = (g1 - g0) / eps
    return J


def gev_delta_cis_raw_and_rescaled(
    z: np.ndarray,
    alpha: float,
    j: float,
    lam: float,
    k: int,
    eps_hess: float = 1e-4,
    eps_jac: float = 1e-6,
) -> Optional[Dict[str, Tuple[float, float]]]:
    fit = fit_gev_mle(z)
    if fit is None:
        return None

    x0 = np.array([fit.c, fit.loc, math.log(fit.scale)], dtype=float)

    f = lambda x: _gev_nll_clog(z, x)
    H = _numerical_hessian(f, x0, eps=eps_hess)

    try:
        cov_x = np.linalg.inv(H)
    except np.linalg.LinAlgError:
        return None

    if not np.isfinite(cov_x).all():
        return None

    # Raw parameter transform:
    # theta_raw = (mu, sigma, xi) = (loc, exp(log_scale), -c)
    sigma_hat = fit.scale
    J_raw = np.array([
        [0.0, 1.0, 0.0],       # d mu / d(c,loc,log_scale)
        [0.0, 0.0, sigma_hat], # d sigma / d(c,loc,log_scale)
        [-1.0, 0.0, 0.0],      # d xi / d(c,loc,log_scale)
    ], dtype=float)

    cov_raw = J_raw @ cov_x @ J_raw.T
    if not np.isfinite(cov_raw).all():
        return None

    # Rescaled parameter transform
    J_star = _jacobian_g_x(z=z, x0=x0, j=j, lam=lam, k=k, eps=eps_jac)
    cov_star = J_star @ cov_x @ J_star.T
    if not np.isfinite(cov_star).all():
        return None

    # Point estimates
    mu_hat = fit.loc
    sigma_hat = fit.scale
    xi_hat = fit.xi
    mu_star_hat, sigma_star_hat, xi_star_hat = rescale_params(
        mu_j=mu_hat,
        sigma_j=sigma_hat,
        xi=xi_hat,
        j=j,
        lam=lam,
        k=k,
    )

    # Standard errors
    se_mu = math.sqrt(max(float(cov_raw[0, 0]), 0.0))
    se_sigma = math.sqrt(max(float(cov_raw[1, 1]), 0.0))
    se_xi = math.sqrt(max(float(cov_raw[2, 2]), 0.0))

    se_mu_star = math.sqrt(max(float(cov_star[0, 0]), 0.0))
    se_sigma_star = math.sqrt(max(float(cov_star[1, 1]), 0.0))
    se_xi_star = math.sqrt(max(float(cov_star[2, 2]), 0.0))

    zcrit = float(norm.ppf(1.0 - alpha / 2.0))

    return {
        # raw
        "mu": (float(mu_hat - zcrit * se_mu), float(mu_hat + zcrit * se_mu)),
        "sigma": (float(sigma_hat - zcrit * se_sigma), float(sigma_hat + zcrit * se_sigma)),
        "xi": (float(xi_hat - zcrit * se_xi), float(xi_hat + zcrit * se_xi)),

        # rescaled
        "mu_star": (
            float(mu_star_hat - zcrit * se_mu_star),
            float(mu_star_hat + zcrit * se_mu_star),
        ),
        "sigma_star": (
            float(sigma_star_hat - zcrit * se_sigma_star),
            float(sigma_star_hat + zcrit * se_sigma_star),
        ),
        "xi_star": (
            float(xi_star_hat - zcrit * se_xi_star),
            float(xi_star_hat + zcrit * se_xi_star),
        ),
    }


def delta_ci_rescaled(
    z: np.ndarray,
    alpha: float,
    j: float,
    lam: float,
    k: int,
    eps_hess: float = 1e-4,
    eps_jac: float = 1e-6,
) -> Optional[Dict[str, Tuple[float, float]]]:
    out = gev_delta_cis_raw_and_rescaled(
        z=z,
        alpha=alpha,
        j=j,
        lam=lam,
        k=k,
        eps_hess=eps_hess,
        eps_jac=eps_jac,
    )
    if out is None:
        return None
    return {
        "mu_star": out["mu_star"],
        "sigma_star": out["sigma_star"],
        "xi": out["xi_star"],
    }

## Algorithm 1 — block length selection

1. candidate block lengths on a grid of spacing `i_step`
2. block maxima, GEV fit, KS / ADF / KPSS diagnostics
3. the admissible set `S`
4. reparameterisation
5. delta-method confidence intervals
6. longest run of consecutive candidates with a common intersection
7. `i*` = the smallest admissible candidate, with a fallback

In [ ]:
# ============================================================
# Longest contiguous run with non-empty common intersection
# ============================================================
def interval_intersection(a: Tuple[float, float], b: Tuple[float, float]) -> Tuple[float, float]:
    return (max(a[0], b[0]), min(a[1], b[1]))


def nonempty(iv: Tuple[float, float]) -> bool:
    return iv[0] <= iv[1]


def longest_overlap_run(idxs_sorted: List[int], ci_by_i: Dict[int, Tuple[float, float]]) -> List[int]:
    best_run: List[int] = []
    best_len = 0
    m = len(idxs_sorted)

    for a in range(m):
        curr = ci_by_i[idxs_sorted[a]]
        if not nonempty(curr):
            continue
        for b in range(a, m):
            if b > a:
                curr = interval_intersection(curr, ci_by_i[idxs_sorted[b]])
            if not nonempty(curr):
                break
            run = idxs_sorted[a:b + 1]
            if len(run) > best_len:
                best_len = len(run)
                best_run = run
    return best_run


# ============================================================
# Selection with diagnostics
# ============================================================
@dataclass
class SelectionDiagnostics:
    df_by_i: pd.DataFrame
    i_star: Optional[int]
    S: List[int]
    S_mu: List[int]
    S_sigma: List[int]
    S_xi: List[int]
    S_all: List[int]


def select_block_size_with_diagnostics(
    y: np.ndarray,
    alpha: float,
    i_min: int,
    N_min: int,
    i_step: int,
    rng: np.random.Generator,
    k: int,
    lam: float,
    i_ref: int,
    eps_hess: float,
    eps_jac: float,
) -> SelectionDiagnostics:
    n = len(y)
    i_max = max([i for i in range(i_min, n + 1) if (n // i) >= N_min], default=i_min)
    I = list(range(i_min, i_max + 1, i_step))

    rows = []
    z_store: Dict[int, np.ndarray] = {}

    # Pass 1: tests + fits + point estimates
    for i in I:
        z = block_maxima(y, i)
        if len(z) < N_min:
            continue
        z_store[i] = z

        p_adf = adf_pvalue(z)
        p_kpss = kpss_pvalue(z)

        fit = fit_gev_mle(z)
        if fit is None:
            rows.append({
                "i": i,
                "n_max": len(z),
                "mu_hat": np.nan,
                "sigma_hat": np.nan,
                "xi_hat": np.nan,
                "mu_star_hat": np.nan,
                "sigma_star_hat": np.nan,
                "p_adf": p_adf,
                "p_kpss": p_kpss,
                "p_ks": 0.0,
                "pass_stationarity": (p_adf <= alpha) or (p_kpss > alpha),
                "pass_ks": False,
            })
            continue

        p_ks = ks_pvalue_gev(z, fit)
        j = i / float(i_ref)

        mu_star_hat, sigma_star_hat, xi_hat_star = rescale_params(
            mu_j=fit.loc,
            sigma_j=fit.scale,
            xi=fit.xi,
            j=j,
            lam=lam,
            k=k,
        )

        rows.append({
            "i": i,
            "n_max": len(z),
            "mu_hat": float(fit.loc),
            "sigma_hat": float(fit.scale),
            "xi_hat": float(fit.xi),
            "mu_star_hat": mu_star_hat,
            "sigma_star_hat": sigma_star_hat,
            "p_adf": p_adf,
            "p_kpss": p_kpss,
            "p_ks": p_ks,
            "pass_stationarity": (p_adf <= alpha) or (p_kpss > alpha),
            "pass_ks": (p_ks > alpha),
        })

    df = pd.DataFrame(rows).sort_values("i").reset_index(drop=True)
    if df.empty:
        return SelectionDiagnostics(df, None, [], [], [], [], [])

    df["pass_all"] = df["pass_stationarity"] & df["pass_ks"]
    S = df.loc[df["pass_all"], "i"].tolist()
    if len(S) == 0:
        return SelectionDiagnostics(df, None, [], [], [], [], [])

    # Pass 2: delta-method CIs for rescaled parameters
    ci_mu: Dict[int, Tuple[float, float]] = {}
    ci_sigma: Dict[int, Tuple[float, float]] = {}
    ci_xi: Dict[int, Tuple[float, float]] = {}

    for i in S:
        z = z_store[i]
        j = i / float(i_ref)
        ci = delta_ci_rescaled(
            z=z,
            alpha=alpha,
            j=j,
            lam=lam,
            k=k,
            eps_hess=eps_hess,
            eps_jac=eps_jac,
        )
        if ci is None:
            continue
        ci_mu[i] = ci["mu_star"]
        ci_sigma[i] = ci["sigma_star"]
        ci_xi[i] = ci["xi"]

    S_ci = sorted(set(ci_mu).intersection(ci_sigma).intersection(ci_xi))
    if len(S_ci) == 0:
        return SelectionDiagnostics(df, None, S, [], [], [], [])

    S_mu = longest_overlap_run(S_ci, ci_mu)
    S_sigma = longest_overlap_run(S_ci, ci_sigma)
    S_xi = longest_overlap_run(S_ci, ci_xi)
    S_all = sorted(set(S_mu).intersection(S_sigma).intersection(S_xi))

    if len(S_all) > 0:
        i_star = min(S_all)
    else:
        fallback = min([
            max(S_mu) if S_mu else math.inf,
            max(S_sigma) if S_sigma else math.inf,
            max(S_xi) if S_xi else math.inf,
        ])
        i_star = int(fallback) if np.isfinite(fallback) else None

    return SelectionDiagnostics(df, i_star, S_ci, S_mu, S_sigma, S_xi, S_all)

## Self-test

Checks the reparameterisation against a direct transcription of Step 4 before any compute is spent.

In [ ]:
def _step4_reference(mu_j, sigma_j, xi, j, lam, k):
    """Literal transcription of Step 4, for checking rescale_params."""
    if abs(xi) < 1e-10:
        return (mu_j - sigma_j * math.log(j) + sigma_j * math.log(lam ** (-(k - 1))),
                sigma_j, 0.0)
    inner = mu_j - sigma_j * ((j ** xi - 1.0) / (xi * j ** xi))
    sig = (lam ** (-(k - 1) * xi)) * sigma_j * (j ** (-xi))
    if xi > 0:
        mu = (lam ** (-(k - 1) * xi)) * inner
    else:
        mu = inner + (sigma_j * j ** (-xi) / xi) * (lam ** (-(k - 1) * xi) - 1.0)
    return mu, sig, xi


_rng = np.random.default_rng(0)
_worst = 0.0
for _ in range(20000):
    _mu = float(_rng.normal(0, 3))
    _sg = float(abs(_rng.normal(1, 0.6)) + 1e-3)
    _xi = float(_rng.choice([_rng.uniform(0.05, 0.8), 0.0, -_rng.uniform(0.05, 0.8)]))
    _j = float(_rng.uniform(1.0, 700.0))
    _k = int(_rng.integers(1, 11))
    _a = np.array(_step4_reference(_mu, _sg, _xi, _j, 2.0, _k))
    _b = np.array(rescale_params(_mu, _sg, _xi, _j, 2.0, _k))
    _worst = max(_worst, float(np.max(np.abs(_a - _b) / np.maximum(np.abs(_a), 1.0))))
assert _worst < 1e-12, f"rescale_params does not match Step 4 (max rel diff {_worst:.2e})"
print(f"Step 4 verified: max relative difference {_worst:.2e}")

## Experiment driver

`make_seed_table` draws from one master RNG in the order
map → observable → orbit. To re-run a subset of observables while keeping the
same orbits, leave `OBSERVABLES` intact and filter `RUN_OBSERVABLES` instead.

In [ ]:
def _fmt(seconds):
    seconds=int(seconds); h,rem=divmod(seconds,3600); m,s=divmod(rem,60)
    return f"{h}h {m:02d}m {s:02d}s" if h else (f"{m}m {s:02d}s" if m else f"{s}s")

MAPS = {
    "doubling": {"T": doubling_map, "p": 0.0, "lambda": doubling_lambda_at(0.0)},
    "logistic": {"T": logistic_map, "p": 0.75, "lambda": logistic_lambda_at(0.75)},
}
OBSERVABLES = [
    ("frechet", 0.3, lambda x, p: phi_frechet(x, p=p, alpha=0.3)),
    ("gumbel", 0.0, lambda x, p: phi_gumbel(x, p=p)),
    ("weibull", -0.3, lambda x, p: phi_weibull(x, p=p, xi=-0.3, C=1.0)),
]

def make_seed_table():
    rng_master = set_seed(SEED); table = {}
    for mn in MAPS:
        for on, _, _ in OBSERVABLES:
            for oid in range(N_ORBITS):
                table[(mn, on, oid)] = int(rng_master.integers(0, 2**32 - 1))
    return table

def run_part1(seed_table):
    os.makedirs(OUT_DIR, exist_ok=True)
    out_csv = os.path.join(OUT_DIR, "results_over_k_all_orbits.csv")
    rows = []; t0 = time.time()
    total = len(MAPS)*len(OBSERVABLES)*N_ORBITS*len(K_VALUES); done = 0
    bar = tqdm(total=total, desc="Part 1 (i* + estimates)", unit="sel", dynamic_ncols=True, smoothing=0.05)
    for map_name, mdd in MAPS.items():
        T, p_target, lam = mdd["T"], mdd["p"], mdd["lambda"]
        for obs_name, xi_true, obs_fn in OBSERVABLES:
            for oid in range(N_ORBITS):
                oseed = seed_table[(map_name, obs_name, oid)]
                rng = np.random.default_rng(oseed)
                orbit = generate_orbit(T=T, n=N, transient=TRANSIENT, rng=rng, noise_std=NOISE_STD)
                phi = obs_fn(orbit, p_target)
                for k in K_VALUES:
                    yk = moving_minimum(phi, k=k)
                    diag = select_block_size_with_diagnostics(
                        y=yk, alpha=ALPHA, i_min=I_MIN, N_min=N_MIN, i_step=I_STEP,
                        rng=rng, k=k, lam=lam, i_ref=I_REF, eps_hess=EPS_HESS, eps_jac=EPS_JAC)
                    istar = diag.i_star; mu=si=xi=mu_s=si_s=np.nan
                    if istar is not None and not diag.df_by_i.empty:
                        r = diag.df_by_i.loc[diag.df_by_i["i"] == istar]
                        if not r.empty:
                            mu=float(r["mu_hat"].iloc[0]); si=float(r["sigma_hat"].iloc[0])
                            xi=float(r["xi_hat"].iloc[0]); mu_s=float(r["mu_star_hat"].iloc[0])
                            si_s=float(r["sigma_star_hat"].iloc[0])
                    m=len(yk); Nk=(m//istar) if istar else np.nan
                    rows.append({"map":map_name,"observable":obs_name,"xi_true":xi_true,
                        "lambda":lam,"orbit_id":oid,"orbit_seed":oseed,"k":k,"i_star":istar,
                        "series_len_m":m,"N_k":Nk,"mu_hat":mu,"sigma_hat":si,"xi_hat":xi,
                        "mu_star_hat":mu_s,"sigma_star_hat":si_s})
                    done += 1
                    bar.set_postfix_str(f"{map_name}/{obs_name} orb{oid+1} k{k}"); bar.update(1)
            pd.DataFrame(rows).to_csv(out_csv, index=False)
            elapsed=time.time()-t0; rate=elapsed/max(done,1); eta=rate*(total-done)
            bar.write(f"[{_fmt(elapsed)}] checkpoint {map_name}/{obs_name} | {done}/{total} "
                      f"({100*done/total:.1f}%) | ~{rate:.1f}s/sel | ETA {_fmt(eta)}")
    bar.close()
    df = pd.DataFrame(rows); df.to_csv(out_csv, index=False)
    print(f"Part 1 done -> {out_csv} ({len(df)} rows) in {_fmt(time.time()-t0)}")
    return df

def run_part2(seed_table):
    os.makedirs(OUT_DIR, exist_ok=True); t0 = time.time()
    total = len(MAPS)*len(OBSERVABLES)*len(K_VALUES); done = 0
    bar = tqdm(total=total, desc="Part 2 (CI curves)", unit="k-sweep", dynamic_ncols=True, smoothing=0.05)
    for map_name, mdd in MAPS.items():
        T, p_target, lam = mdd["T"], mdd["p"], mdd["lambda"]
        for obs_name, xi_true, obs_fn in OBSERVABLES:
            oseed = seed_table[(map_name, obs_name, CI_ORBIT_ID)]
            rng = np.random.default_rng(oseed)
            orbit = generate_orbit(T=T, n=N, transient=TRANSIENT, rng=rng, noise_std=NOISE_STD)
            phi = obs_fn(orbit, p_target); rows = []
            for k in K_VALUES:
                yk = moving_minimum(phi, k=k)
                diag = select_block_size_with_diagnostics(
                    y=yk, alpha=ALPHA, i_min=I_MIN, N_min=N_MIN, i_step=I_STEP,
                    rng=rng, k=k, lam=lam, i_ref=I_REF, eps_hess=EPS_HESS, eps_jac=EPS_JAC)
                istar = diag.i_star
                cand = diag.df_by_i["i"].tolist()
                for i in tqdm(cand, desc=f"  {map_name}/{obs_name} k={k}", unit="i", leave=False, dynamic_ncols=True):
                    z = block_maxima(yk, i)
                    if len(z) < N_MIN: continue
                    ci = gev_delta_cis_raw_and_rescaled(z=z, alpha=ALPHA, j=i/float(I_REF),
                        lam=lam, k=k, eps_hess=EPS_HESS, eps_jac=EPS_JAC)
                    if ci is None: continue
                    r = diag.df_by_i.loc[diag.df_by_i["i"] == i]
                    rows.append({"k":k,"i":i,"i_star":istar,
                        "mu":float(r["mu_hat"].iloc[0]),"sigma":float(r["sigma_hat"].iloc[0]),
                        "xi":float(r["xi_hat"].iloc[0]),"mu_s":float(r["mu_star_hat"].iloc[0]),
                        "sigma_s":float(r["sigma_star_hat"].iloc[0]),
                        "mu_lo":ci["mu"][0],"mu_hi":ci["mu"][1],
                        "sigma_lo":ci["sigma"][0],"sigma_hi":ci["sigma"][1],
                        "xi_lo":ci["xi"][0],"xi_hi":ci["xi"][1],
                        "mu_s_lo":ci["mu_star"][0],"mu_s_hi":ci["mu_star"][1],
                        "sigma_s_lo":ci["sigma_star"][0],"sigma_s_hi":ci["sigma_star"][1],
                        "xi_true":xi_true,"lambda":lam})
                done += 1
                eta=(time.time()-t0)/max(done,1)*(total-done); bar.set_postfix_str(f"ETA {_fmt(eta)}"); bar.update(1)
            out = os.path.join(OUT_DIR, f"ci_curves_{map_name}_{obs_name}_orb{CI_ORBIT_ID+1}.parquet")
            dfc = pd.DataFrame(rows)
            try: dfc.to_parquet(out, index=False)
            except Exception:
                out = out.replace(".parquet",".csv"); dfc.to_csv(out, index=False)
            bar.write(f"[{_fmt(time.time()-t0)}] Part2: {map_name}/{obs_name} -> {os.path.basename(out)} ({len(dfc)} rows)")
    bar.close()
    print(f"Part 2 done in {_fmt(time.time()-t0)}")

# ---------------------------------------------------------------------
# Load stored results if present; otherwise run the full computation.
# ---------------------------------------------------------------------

_part1_csv = os.path.join(OUT_DIR, "results_over_k_all_orbits.csv")
_part2_any = glob.glob(os.path.join(OUT_DIR, "ci_curves_*"))

## Run

In [ ]:
seed_table = make_seed_table()
results_df = run_part1(seed_table)
run_part2(seed_table)
print("ANALYSIS COMPLETE ->", OUT_DIR)

## Save

Zip the results for the figures notebook.

In [ ]:
import shutil

shutil.make_archive("ch4_results", "zip", ".", OUT_DIR)
print("wrote ch4_results.zip")
try:
    from google.colab import files
    files.download("ch4_results.zip")
except Exception:
    pass